### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.metrics import accuracy_score, log_loss

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor
from src.gb_classifier import GBClassifier

### CONFIGURATION

In [2]:
active_dataset = "mnist"

data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config(active_dataset)

active_dataset_config = datasets_config[active_dataset]
problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_image_data(
    active_dataset
)

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)
X_test, y_test = processor.split_features_target(test)

y_train, y_valid, y_test = processor.transform_target(y_train, y_valid, y_test)

In [4]:
def total_parameters(N_1: int, N_2: int, C: int, M: int, H: int, W: int) -> int:
    """Calculate the total trainable parameters.

    The architecture consists of two sequential 5x5 convolutional layers,
    each followed by a 2x2 MaxPool stride reduction step, feeding into a 
    hidden linear layer before reaching a 10-class output head.

    Args:
        N_1 (int): Number of filters in the first convolutional stage.
        N_2 (int): Number of filters in the second convolutional stage.
        C (int): Input channel depth dimension.
        M (int): Size of the dense hidden linear layer.
        H (int): Vertical spatial pixel height boundary.
        W (int): Horizontal spatial pixel width boundary.

    Returns:
        int: Calculated summation of weights and biases across all layers.
    """
    D = ((((H - 4) / 2) - 4) / 2) * (((((W - 4) / 2) - 4) / 2)) * N_2
    return int(N_1 * (25 * C + 1) + N_2 * (25 * N_1 + 1) + M * (D + 1) + 10 * (M + 1))

print("BIG CNN:", total_parameters(16, 32, 1, 64, 28, 28))
print("SMALL CNN:", total_parameters(4, 8, 1, 16, 28, 28))

BIG CNN: 46730
SMALL CNN: 3146


### BIG CNN

Trainable parameters: 46,730

In [ ]:
class CNN(CNNRegressor):
    """Convolutional Neural Network classifier with early stopping tracking.

    Inherits network structure construction from CNNRegressor while adapting 
    the dimension tracking calculations and criteria for classification tasks.
    """

    def __init__(self, **hyperparameters):
        """Initializes base architectural configurations dynamically."""
        super().__init__(**hyperparameters)

    def fit(
        self, 
        X_train: np.ndarray, 
        y_train: np.ndarray, 
        X_valid: np.ndarray, 
        y_valid: np.ndarray, 
        patience: int = 10
    ) -> None:
        """Calculates adaptive layer limits and trains the network on Cross-Entropy.

        Args:
            X_train (np.ndarray): Training images in (N, C, H, W) layout.
            y_train (np.ndarray): Categorical target training labels.
            X_valid (np.ndarray): Validation images in (N, C, H, W) layout.
            y_valid (np.ndarray): Validation categorical target labels.
            patience (int): Iteration limit allowed without validation gain.
        """
        in_channels = X_train.shape[1]
        output_size = int(np.max(y_train)) + 1

        image_size = X_train.shape[-1]

        conv1_out = image_size - self.kernel_size + 1
        pool1_out = conv1_out // self.pool_size

        conv2_out = pool1_out - self.kernel_size + 1
        pool2_out = conv2_out // self.pool_size

        linear_input = self.channels[1] * pool2_out * pool2_out
        
        self._get_network(in_channels, linear_input, output_size)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            print(f"Epoch: {epoch + 1} | Validation Log Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predicts crisp class labels for the input images.

        Args:
            X (np.ndarray): Image tensor in matrix array layout.

        Returns:
            np.ndarray: Predicted class labels mapping to class indices.
        """
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Computes probability distributions across image categories using Softmax.

        Args:
            X (np.ndarray): Image tensor in matrix array layout.

        Returns:
            np.ndarray: Matrix containing row-wise prediction probability arrays.
        """
        X_t = torch.from_numpy(X).to(torch.float32)
        X_t = X_t.to(self.device)

        self.eval()
        with torch.no_grad():
            predictions = self.forward(X_t)
        
        return torch.softmax(predictions, dim=1).cpu().numpy()

In [22]:
%%time

big_cnn = CNN(epochs=100, learning_rate=0.001, channels=[16, 32], kernel_size=5, pool_size=2, hidden_size=64, batch_size=256)
big_cnn.fit(X_train, y_train, X_valid, y_valid)

big_cnn_valid_preds = big_cnn.predict(X_valid)
big_cnn_valid_probs = big_cnn.predict_proba(X_valid)

big_cnn_test_preds = big_cnn.predict(X_test)
big_cnn_test_probs = big_cnn.predict_proba(X_test)

print('-' * 50)
print(f"BIG CNN validation log loss: {log_loss(y_valid, big_cnn_valid_probs):.4f}")
print(f"BIG CNN validation accuracy: {accuracy_score(y_valid, big_cnn_valid_preds):.4f}")
print('-' * 50)
print(f"BIG CNN test log loss: {log_loss(y_test, big_cnn_test_probs):.4f}")
print(f"BIG CNN test accuracy: {accuracy_score(y_test, big_cnn_test_preds):.4f}")
print('-' * 50)

Epoch: 1 | Validation Log Loss: 0.1637 | Validation Accuracy: 0.9530
Epoch: 2 | Validation Log Loss: 0.1020 | Validation Accuracy: 0.9702
Epoch: 3 | Validation Log Loss: 0.0703 | Validation Accuracy: 0.9797
Epoch: 4 | Validation Log Loss: 0.0642 | Validation Accuracy: 0.9823
Epoch: 5 | Validation Log Loss: 0.0545 | Validation Accuracy: 0.9837
Epoch: 6 | Validation Log Loss: 0.0545 | Validation Accuracy: 0.9847
Epoch: 7 | Validation Log Loss: 0.0481 | Validation Accuracy: 0.9858
Epoch: 8 | Validation Log Loss: 0.0482 | Validation Accuracy: 0.9867
Epoch: 9 | Validation Log Loss: 0.0478 | Validation Accuracy: 0.9873
Epoch: 10 | Validation Log Loss: 0.0476 | Validation Accuracy: 0.9870
Epoch: 11 | Validation Log Loss: 0.0435 | Validation Accuracy: 0.9882
Epoch: 12 | Validation Log Loss: 0.0388 | Validation Accuracy: 0.9882
Epoch: 13 | Validation Log Loss: 0.0480 | Validation Accuracy: 0.9876
Epoch: 14 | Validation Log Loss: 0.0402 | Validation Accuracy: 0.9876
Epoch: 15 | Validation Log Lo

### GRADIENT BOOSTING (BIG CNN)

In [30]:
big_cnn_gb = GBClassifier(**gradient_boosting_config, weak_learner_config=weak_learner_config)
big_cnn_gb.load_model("models/mnist/2026_04_29_10_15/model.joblib")

big_cnn_gb_valid_preds = big_cnn_gb.predict(X_valid)
big_cnn_gb_valid_probs = big_cnn_gb.predict_proba(X_valid)

big_cnn_gb_test_preds = big_cnn_gb.predict(X_test)
big_cnn_gb_test_probs = big_cnn_gb.predict_proba(X_test)

print('-' * 50)
print(f"GB (BIG CNN) validation log loss: {log_loss(y_valid, big_cnn_gb_valid_probs):.4f}")
print(f"GB (BIG CNN) validation accuracy: {accuracy_score(y_valid, big_cnn_gb_valid_preds):.4f}")
print('-' * 50)
print(f"GB (BIG CNN) test log loss: {log_loss(y_test, big_cnn_gb_test_probs):.4f}")
print(f"GB (BIG CNN) test accuracy: {accuracy_score(y_test, big_cnn_gb_test_preds):.4f}")

2026-04-29 10:21:02,002 - INFO - Model loaded from models/mnist/2026_04_29_10_15/model.joblib


--------------------------------------------------
GB (BIG CNN) validation log loss: 0.0545
GB (BIG CNN) validation accuracy: 0.9935
--------------------------------------------------
GB (BIG CNN) test log loss: 0.0502
GB (BIG CNN) test accuracy: 0.9942


### SMALL CNN

Trainable parameters: 3,146

In [7]:
%%time

small_cnn = CNN(epochs=100, learning_rate=0.001, channels=[4, 8], kernel_size=5, pool_size=2, hidden_size=16, batch_size=256)
small_cnn.fit(X_train, y_train, X_valid, y_valid)

small_cnn_valid_preds = small_cnn.predict(X_valid)
small_cnn_valid_probs = small_cnn.predict_proba(X_valid)

small_cnn_test_preds = small_cnn.predict(X_test)
small_cnn_test_probs = small_cnn.predict_proba(X_test)

print('-' * 50)
print(f"SMALL CNN validation log loss: {log_loss(y_valid, small_cnn_valid_probs):.4f}")
print(f"SMALL CNN validation accuracy: {accuracy_score(y_valid, small_cnn_valid_preds):.4f}")
print('-' * 50)
print(f"SMALL CNN test log loss: {log_loss(y_test, small_cnn_test_probs):.4f}")
print(f"SMALL CNN test accuracy: {accuracy_score(y_test, small_cnn_test_preds):.4f}")
print('-' * 50)

Epoch: 1 | Validation Log Loss: 0.3967 | Validation Accuracy: 0.8827
Epoch: 2 | Validation Log Loss: 0.2612 | Validation Accuracy: 0.9212
Epoch: 3 | Validation Log Loss: 0.1857 | Validation Accuracy: 0.9435
Epoch: 4 | Validation Log Loss: 0.1523 | Validation Accuracy: 0.9542
Epoch: 5 | Validation Log Loss: 0.1264 | Validation Accuracy: 0.9621
Epoch: 6 | Validation Log Loss: 0.1164 | Validation Accuracy: 0.9647
Epoch: 7 | Validation Log Loss: 0.1115 | Validation Accuracy: 0.9679
Epoch: 8 | Validation Log Loss: 0.1026 | Validation Accuracy: 0.9682
Epoch: 9 | Validation Log Loss: 0.0971 | Validation Accuracy: 0.9708
Epoch: 10 | Validation Log Loss: 0.0938 | Validation Accuracy: 0.9715
Epoch: 11 | Validation Log Loss: 0.0840 | Validation Accuracy: 0.9753
Epoch: 12 | Validation Log Loss: 0.0827 | Validation Accuracy: 0.9747
Epoch: 13 | Validation Log Loss: 0.0815 | Validation Accuracy: 0.9753
Epoch: 14 | Validation Log Loss: 0.0830 | Validation Accuracy: 0.9751
Epoch: 15 | Validation Log Lo

### GRADIENT BOOSTING (SMALL CNN)

In [9]:
small_cnn_gb = GBClassifier(**gradient_boosting_config, weak_learner_config=weak_learner_config)
small_cnn_gb.load_model("models/mnist/2026_04_29_15_01/model.joblib")

small_cnn_gb_valid_preds = small_cnn_gb.predict(X_valid)
small_cnn_gb_valid_probs = small_cnn_gb.predict_proba(X_valid)

small_cnn_gb_test_preds = small_cnn_gb.predict(X_test)
small_cnn_gb_test_probs = small_cnn_gb.predict_proba(X_test)

print('-' * 50)
print(f"GB (SMALL CNN) validation log loss: {log_loss(y_valid, small_cnn_gb_valid_probs):.4f}")
print(f"GB (SMALL CNN) validation accuracy: {accuracy_score(y_valid, small_cnn_gb_valid_preds):.4f}")
print('-' * 50)
print(f"GB (SMALL CNN) test log loss: {log_loss(y_test, small_cnn_gb_test_probs):.4f}")
print(f"GB (SMALL CNN) test accuracy: {accuracy_score(y_test, small_cnn_gb_test_preds):.4f}")

2026-04-29 15:02:09,880 - INFO - Model loaded from models/mnist/2026_04_29_15_01/model.joblib


--------------------------------------------------
GB (SMALL CNN) validation log loss: 0.0849
GB (SMALL CNN) validation accuracy: 0.9891
--------------------------------------------------
GB (SMALL CNN) test log loss: 0.0779
GB (SMALL CNN) test accuracy: 0.9912
